# Chapter 09 - Categorical Feature Engineering

Most machine learning algorithms require numerical input. Categorical features — colors, cities,
education levels — must be converted to numbers before a model can use them. The choice of
encoding strategy significantly affects model performance, and getting it wrong can introduce
artificial ordinal relationships or blow up your feature space.

**Topics covered:**
- Ordinal vs nominal categorical data
- Label encoding with LabelEncoder
- One-hot encoding with OneHotEncoder and pd.get_dummies()
- When to use which encoding
- High-cardinality features: frequency encoding and target encoding
- Handling unknown categories at prediction time
- Binary encoding for boolean features
- Practical example: encoding a dataset with mixed types

## Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

np.random.seed(42)

## Types of Categorical Data

Categorical features come in two flavors:

### Nominal (No Natural Order)
The categories have no inherent ranking. Assigning numbers like 0, 1, 2 would imply
a false order.

- Colors: red, blue, green
- Cities: London, Paris, Tokyo
- Blood type: A, B, AB, O

### Ordinal (Natural Order Exists)
The categories have a meaningful ranking, but the intervals between them may not be equal.

- Education: high school < bachelor < master < PhD
- T-shirt size: S < M < L < XL
- Satisfaction: poor < fair < good < excellent

In [ ]:
# Create a sample dataset with both types
data = pd.DataFrame({
    'color': np.random.choice(['red', 'blue', 'green', 'yellow'], size=10),
    'size': np.random.choice(['S', 'M', 'L', 'XL'], size=10),
    'education': np.random.choice(['high_school', 'bachelor', 'master', 'phd'], size=10),
    'city': np.random.choice(['London', 'Paris', 'Tokyo', 'Sydney'], size=10),
    'price': np.random.uniform(10, 100, size=10).round(2),
})

print('Nominal columns: color, city')
print('Ordinal columns: size, education')
print('Numerical column: price')
print()
data

## Label Encoding

`LabelEncoder` converts each unique category to an integer (0, 1, 2, ...).

**When to use it:**
- For **ordinal** features where the integer mapping respects the natural order
- For **target variables** (classification labels)

**When NOT to use it:**
- For **nominal** features fed into linear models or KNN — the model would interpret
  the numbers as having an order (e.g., "Tokyo=3 > London=1")

In [ ]:
le = LabelEncoder()
data['color_encoded'] = le.fit_transform(data['color'])

print('Classes:', le.classes_)
print('Mapping:')
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f'  {cls} -> {idx}')
print()
data[['color', 'color_encoded']]

In [ ]:
# Inverse transform: go back from numbers to labels
original = le.inverse_transform(data['color_encoded'])
print('Inverse transform:', original)

### OrdinalEncoder for Ordered Categories

When the order matters, use `OrdinalEncoder` with an explicit category ordering.
This ensures that the integer mapping matches the real-world ranking.

In [ ]:
size_order = [['S', 'M', 'L', 'XL']]
edu_order = [['high_school', 'bachelor', 'master', 'phd']]

ord_enc = OrdinalEncoder(categories=size_order)
data['size_encoded'] = ord_enc.fit_transform(data[['size']])

edu_enc = OrdinalEncoder(categories=edu_order)
data['edu_encoded'] = edu_enc.fit_transform(data[['education']])

data[['size', 'size_encoded', 'education', 'edu_encoded']]

> The encoding respects the natural order: S=0 < M=1 < L=2 < XL=3.
> Without specifying `categories`, OrdinalEncoder would use alphabetical order,
> which gives L=0 < M=1 < S=2 < XL=3 — completely wrong.

## One-Hot Encoding

One-hot encoding creates a **binary column for each category**. This is the standard
approach for **nominal** features because it introduces no artificial ordering.

For a feature with *k* categories, one-hot encoding creates *k* new binary columns
(or *k-1* if you drop one to avoid multicollinearity).

### Using sklearn's OneHotEncoder

In [ ]:
ohe = OneHotEncoder(sparse_output=False, dtype=int)
color_ohe = ohe.fit_transform(data[['color']])

print('Feature names:', ohe.get_feature_names_out())
print()

color_df = pd.DataFrame(color_ohe, columns=ohe.get_feature_names_out(), index=data.index)
pd.concat([data[['color']], color_df], axis=1)

In [ ]:
# drop='first' avoids the dummy variable trap (multicollinearity)
# With k categories, k-1 columns carry the same information
ohe_drop = OneHotEncoder(sparse_output=False, dtype=int, drop='first')
color_dropped = ohe_drop.fit_transform(data[['color']])

print('With drop="first":')
print('Feature names:', ohe_drop.get_feature_names_out())
print(f'Columns: {color_ohe.shape[1]} -> {color_dropped.shape[1]}')

### Using pd.get_dummies()

Pandas provides a convenient function that does the same thing. It is great for
quick exploration but has an important limitation: it does not store the encoding
mapping, so it can produce different columns on new data.

In [ ]:
dummies = pd.get_dummies(data[['color', 'city']], prefix=['color', 'city'])
dummies.head()

In [ ]:
# drop_first=True to avoid multicollinearity
dummies_dropped = pd.get_dummies(data[['color', 'city']], drop_first=True)
print(f'Columns with drop_first: {list(dummies_dropped.columns)}')
dummies_dropped.head()

### When to Use Which

| Method | Pros | Cons | Best for |
|---|---|---|---|
| `pd.get_dummies()` | Simple, returns DataFrame | No memory of categories; unsafe for production | Quick exploration, EDA |
| `OneHotEncoder` | Remembers categories, handles unknowns, fits in pipelines | Returns array (needs extra step for DataFrame) | Production code, pipelines |
| `OrdinalEncoder` | Preserves order, single column output | Implies ordering even when inappropriate | Ordinal features, tree-based models |
| `LabelEncoder` | Simple | Only for 1D (target labels), implies ordering | Encoding target variable |

## Handling High-Cardinality Features

One-hot encoding a feature with hundreds or thousands of unique values (e.g., zip codes,
product IDs) creates a massive sparse matrix. This slows training and can cause overfitting.

Here are practical alternatives:

### Frequency Encoding

Replace each category with its frequency (count or proportion) in the training set.
This captures the idea that common categories may behave differently from rare ones.

In [ ]:
# Simulate a high-cardinality feature
np.random.seed(42)
cities = np.random.choice(
    ['London', 'Paris', 'Tokyo', 'Sydney', 'Berlin', 'Rome', 'Madrid',
     'Oslo', 'Lisbon', 'Dublin', 'Prague', 'Vienna', 'Zurich', 'Athens'],
    size=200,
    p=[0.20, 0.15, 0.12, 0.10, 0.08, 0.07, 0.06,
       0.05, 0.04, 0.04, 0.03, 0.03, 0.02, 0.01]
)
df_cities = pd.DataFrame({'city': cities})

# Frequency encoding
freq_map = df_cities['city'].value_counts(normalize=True)
df_cities['city_freq'] = df_cities['city'].map(freq_map)

print('Frequency mapping:')
for city, freq in freq_map.items():
    print(f'  {city:>8s}: {freq:.3f}')

print()
df_cities.head(10)

### Target Encoding (Concept)

Target encoding replaces each category with the **mean of the target variable** for that
category. For example, if houses in London have an average price of 450k, then "London"
becomes 450000.

**Advantages:**
- Single column output regardless of cardinality
- Captures the relationship between the category and the target

**Dangers:**
- High risk of **data leakage** and **overfitting**, especially for rare categories
- Must be computed on training folds only (use within cross-validation)

Scikit-learn provides `TargetEncoder` (added in version 1.3) which handles these
concerns with built-in regularization.

In [ ]:
# Manual target encoding demonstration (for illustration only)
np.random.seed(42)
df_target = pd.DataFrame({
    'city': np.random.choice(['London', 'Paris', 'Tokyo'], size=100),
    'price': np.random.normal(loc=300, scale=50, size=100)
})

# Add a city-dependent signal
city_premium = {'London': 150, 'Paris': 100, 'Tokyo': 50}
df_target['price'] += df_target['city'].map(city_premium)

# Compute target encoding (on training data only in practice)
target_means = df_target.groupby('city')['price'].mean()
df_target['city_target_enc'] = df_target['city'].map(target_means)

print('Target encoding:')
print(target_means.round(2).to_string())
print()
df_target.head()

> **Warning:** The example above computes target encoding on the entire dataset for simplicity.
> In a real project, always compute it on training data only and apply the mapping to the
> test set. Better yet, use scikit-learn's `TargetEncoder` inside a pipeline.

## Handling Unknown Categories at Prediction Time

When your model encounters a category at prediction time that it never saw during training,
one-hot encoding will fail by default. The `handle_unknown` parameter controls this behavior.

In [ ]:
# Simulate: train on 3 colors, encounter a new one at prediction time
train_colors = pd.DataFrame({'color': ['red', 'blue', 'green', 'red', 'blue']})
new_colors = pd.DataFrame({'color': ['red', 'purple']})  # 'purple' is unknown

# Default behavior: will raise an error
ohe_strict = OneHotEncoder(sparse_output=False)
ohe_strict.fit(train_colors)

try:
    ohe_strict.transform(new_colors)
except ValueError as e:
    print(f'Error: {e}')

In [ ]:
# Solution: handle_unknown='ignore' encodes unknown categories as all zeros
ohe_safe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_safe.fit(train_colors)

result = ohe_safe.transform(new_colors)
print('Feature names:', ohe_safe.get_feature_names_out())
print()
print(pd.DataFrame(result, columns=ohe_safe.get_feature_names_out(), index=['red', 'purple (unknown)']))

> The unknown category "purple" is encoded as all zeros — it does not match any known category.
> This is a safe default for production systems where new categories can appear.

## Binary Encoding for Boolean Features

Boolean features (yes/no, true/false, male/female) are the simplest case. They can be
encoded as a single 0/1 column.

In [ ]:
df_bool = pd.DataFrame({
    'has_garage': ['yes', 'no', 'yes', 'yes', 'no'],
    'is_furnished': [True, False, True, False, True],
    'pet_friendly': ['Y', 'N', 'N', 'Y', 'Y'],
})

# String booleans: use map
df_bool['has_garage_enc'] = df_bool['has_garage'].map({'yes': 1, 'no': 0})

# Python booleans: cast directly
df_bool['is_furnished_enc'] = df_bool['is_furnished'].astype(int)

# Single-character codes: use map
df_bool['pet_friendly_enc'] = df_bool['pet_friendly'].map({'Y': 1, 'N': 0})

df_bool

> **Tip:** For binary features, there is no need for one-hot encoding. A single column
> with values 0 and 1 captures all the information. One-hot encoding would create two
> perfectly correlated columns, which is redundant.

## Practical Example: Encoding a Mixed-Type Dataset

Let us build a realistic dataset and apply the right encoding to each column.

In [ ]:
np.random.seed(42)
n = 200

houses = pd.DataFrame({
    'area_sqft': np.random.normal(1500, 400, n).round(0),
    'bedrooms': np.random.choice([1, 2, 3, 4, 5], n, p=[0.1, 0.25, 0.35, 0.2, 0.1]),
    'neighborhood': np.random.choice(['downtown', 'suburbs', 'rural'], n, p=[0.4, 0.4, 0.2]),
    'condition': np.random.choice(['poor', 'fair', 'good', 'excellent'], n),
    'has_pool': np.random.choice([True, False], n),
    'style': np.random.choice(['colonial', 'modern', 'ranch', 'victorian'], n),
})

# Generate a price target with some signal
base_price = 200_000
houses['price'] = (
    base_price
    + houses['area_sqft'] * 100
    + houses['bedrooms'] * 15_000
    + houses['neighborhood'].map({'downtown': 50_000, 'suburbs': 20_000, 'rural': -10_000})
    + houses['condition'].map({'poor': -30_000, 'fair': 0, 'good': 20_000, 'excellent': 40_000})
    + houses['has_pool'] * 25_000
    + np.random.normal(0, 20_000, n)
).round(0)

print(f'Shape: {houses.shape}')
houses.head()

In [ ]:
# Identify column types
print('Column analysis:')
print(f'  Numerical:  area_sqft, bedrooms')
print(f'  Ordinal:    condition (poor < fair < good < excellent)')
print(f'  Nominal:    neighborhood, style')
print(f'  Binary:     has_pool')
print(f'  Target:     price')

In [ ]:
houses_encoded = houses.copy()

# 1. Binary: simple integer cast
houses_encoded['has_pool'] = houses_encoded['has_pool'].astype(int)

# 2. Ordinal: OrdinalEncoder with explicit order
condition_order = [['poor', 'fair', 'good', 'excellent']]
ord_enc = OrdinalEncoder(categories=condition_order)
houses_encoded['condition'] = ord_enc.fit_transform(houses_encoded[['condition']])

# 3. Nominal: One-hot encoding
nominal_cols = ['neighborhood', 'style']
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
nominal_encoded = ohe.fit_transform(houses_encoded[nominal_cols])
nominal_df = pd.DataFrame(
    nominal_encoded,
    columns=ohe.get_feature_names_out(),
    index=houses_encoded.index
)

# Combine everything
houses_final = pd.concat([
    houses_encoded.drop(columns=nominal_cols),
    nominal_df
], axis=1)

print(f'Columns before: {houses.shape[1]}')
print(f'Columns after:  {houses_final.shape[1]}')
print(f'\nAll columns: {list(houses_final.columns)}')
houses_final.head()

In [ ]:
# Verify: all columns are now numerical
print('Data types:')
print(houses_final.dtypes.to_string())
print()
print('Any non-numeric columns:', houses_final.select_dtypes(exclude='number').columns.tolist())

## Common Pitfalls

1. **Using LabelEncoder on nominal features in a linear model:** The model interprets
   "Tokyo=2" as twice "London=1". Use one-hot encoding instead.

2. **Fitting the encoder on the full dataset before splitting:** This leaks information
   from the test set into the encoder. Always fit on training data only.

3. **Forgetting `handle_unknown='ignore'`:** Your model will crash in production when
   it encounters a new category.

4. **One-hot encoding high-cardinality features:** A zip code column with 40,000 unique
   values creates 40,000 columns. Use frequency encoding or target encoding instead.

5. **Using `pd.get_dummies()` in production:** It does not remember the training categories.
   If the test set is missing a category, the column will be absent, breaking the model.

## Encoding Decision Guide

| Feature type | Cardinality | Recommended encoding |
|---|---|---|
| Binary (yes/no) | 2 | Single 0/1 column |
| Ordinal | Low | OrdinalEncoder with specified order |
| Nominal | Low (< 10-15) | OneHotEncoder (drop='first') |
| Nominal | Medium (15-100) | OneHotEncoder or frequency encoding |
| Nominal | High (100+) | Frequency encoding or target encoding |
| Target variable | Any | LabelEncoder |

---

**Summary:** This notebook covered how to convert categorical features into numerical
representations that machine learning models can use. The key distinction is between
ordinal features (where integer encoding preserves meaningful order) and nominal features
(where one-hot encoding avoids introducing false relationships). For high-cardinality
features, frequency encoding and target encoding offer practical alternatives. Always
use `handle_unknown='ignore'` in production and fit encoders on training data only.

**Next up:** Putting it all together with Pipelines and ColumnTransformers.